In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
df1 = pd.read_csv("../../tmp_data/bek_processed_part_2.csv")
df2 = pd.read_csv("../../tmp_data/vanya_processed_part_2.csv")

In [3]:
df = pd.merge(df1, df2, on="id", how="inner")


# Считаем медианную площадь по каждому rooms_count
area_by_rooms = df.groupby('rooms_count')['total_area'].median()

# Заполняем NaN через ближайшую медианную площадь
mask = df['rooms_count'].isna()
df.loc[mask, 'rooms_count'] = df.loc[mask, 'total_area'].apply(
    lambda area: (area_by_rooms - area).abs().idxmin()
)

print(df['rooms_count'].isna().sum())

0


In [4]:
# Процент пропусков
print("Процент пропусков в living_area: ", df["living_area"].isna().mean() * 100)
print("Процент пропусков в kitchen_area: ", df["kitchen_area"].isna().mean() * 100)

# Флаги до заполнения
df["living_area_known"] = df["living_area"].notna().astype(int)
df["kitchen_area_known"] = df["kitchen_area"].notna().astype(int)

# Заполняем медианой по rooms_count + house_type
for col in ["living_area", "kitchen_area"]:
    df[col] = df.groupby(["rooms_count", "house_type"])[col]\
        .transform(lambda x: x.fillna(x.median()))
    # Fallback на только rooms_count
    df[col] = df.groupby("rooms_count")[col]\
        .transform(lambda x: x.fillna(x.median()))
    # Fallback на глобальную медиану
    df[col] = df[col].fillna(df[col].median())

# Фильтруем после заполнения
df = df[df["living_area"] < df["total_area"]]
df = df[df["kitchen_area"] < df["total_area"]]

print(df[["living_area", "kitchen_area"]].isna().sum())

Процент пропусков в living_area:  12.717223082025205
Процент пропусков в kitchen_area:  22.72385584788857
living_area     0
kitchen_area    0
dtype: int64


In [5]:
df.to_csv("../../tmp_data/merged_part_2.csv", index=False)

df

,id,ceiling_height,metro_minutes,metro_walking,total_area,living_area,kitchen_area,price,utilities_amount,utilities_included,prepayment_months,is_long_rental_term,ceiling_height_known,rooms_count,types_room,floor,house_floor,house_type,parking,renovation,count_loggia,count_balcony,view_of_courtyard,view_of_street,combined_bathroom_count,separate_bathroom_count,is_child,is_pet,room_furniture,kitchen_furniture,bath,shower_cabin,washing_machine,air_conditioner,dishwasher,tv,fridge,internet,telephone,count_passenger_lift,count_freight_lift,is_garbage_chute,living_area_known,kitchen_area_known
0,271271157,3.00,9,1,200.0,20.0,17.0,500000.0,0.0,1,1.0,1,1,4.0,NaN,5,16,9.0,4.0,3.0,NaN,NaN,NaN,NaN,1.0,0.0,1,1,1,1,1,1.0,1,1.0,1.0,1.0,1,1,1,4.0,1.0,0.0,1,0
1,271634126,3.50,8,1,198.0,95.0,18.0,500000.0,0.0,1,1.0,1,1,4.0,NaN,5,16,8.0,4.0,3.0,NaN,NaN,1.0,1.0,2.0,1.0,1,0,1,1,1,1.0,1,1.0,1.0,1.0,1,1,0,1.0,1.0,0.0,1,1
2,271173086,3.20,7,1,200.0,81.0,15.0,500000.0,0.0,1,1.0,1,1,4.0,", Оба варианта",5,16,NaN,4.0,2.0,NaN,NaN,1.0,1.0,3.0,0.0,1,0,1,1,1,1.0,1,1.0,1.0,1.0,1,1,1,1.0,0.0,NaN,1,1
3,272197456,3.20,3,1,170.0,81.0,15.0,400000.0,0.0,1,1.0,1,1,4.0,", Оба варианта",5,6,NaN,4.0,2.0,NaN,NaN,1.0,1.0,3.0,0.0,0,1,1,1,1,1.0,1,1.0,1.0,1.0,1,1,1,1.0,0.0,NaN,1,1
4,273614615,2.64,7,1,58.0,38.0,5.0,225000.0,0.0,1,1.0,1,1,2.0,NaN,12,26,0.0,NaN,2.0,NaN,NaN,1.0,1.0,2.0,0.0,0,0,1,1,1,1.0,1,0.0,1.0,1.0,1,1,0,1.0,1.0,1.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22610,215565511,2.64,8,1,35.0,19.0,9.0,42000.0,0.0,1,1.0,1,0,1.0,NaN,10,14,NaN,NaN,2.0,0.0,1.0,NaN,NaN,1.0,0.0,0,0,1,1,1,0.0,1,1.0,0.0,0.0,1,1,0,1.0,1.0,NaN,1,1
22611,274654844,2.64,7,1,38.7,16.5,11.0,45000.0,0.0,1,1.0,1,0,1.0,NaN,5,18,9.0,NaN,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0,0,1,1,1,0.0,1,0.0,0.0,1.0,1,0,0,1.0,1.0,0.0,1,1
22612,268679909,2.64,6,1,43.1,31.0,8.0,50000.0,0.0,1,1.0,1,0,2.0,", Оба варианта",5,5,5.0,NaN,3.0,0.0,1.0,1.0,1.0,1.0,0.0,1,0,0,1,0,1.0,1,1.0,0.0,0.0,1,1,0,0.0,0.0,1.0,0,0
22613,274807525,2.65,11,1,52.5,10.0,11.0,55000.0,0.0,1,2.0,1,1,2.0,NaN,8,23,9.0,0.0,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0,0,1,1,1,0.0,1,1.0,1.0,0.0,1,0,0,3.0,0.0,0.0,1,0
